# Explore a single skin

Edita `ITEM_NAME` abaixo e corre todas as células. Mostra:
- Profundidade e granularidade da histórico de preço
- Preços atuais por marketplace
- Estatísticas mensais e anuais
- Plot interativo

In [1]:
ITEM_NAME = "AK-47 | Redline (Field-Tested)"

import polars as pl

item_df = (
    pl.scan_parquet("../data/items.parquet")
      .filter(pl.col("name") == ITEM_NAME)
      .collect()
)
assert item_df.height, f"Not found: {ITEM_NAME}"
item = item_df.row(0, named=True)
print({k: type(v).__name__ for k, v in item.items()})

{'hash_name': 'str', 'name': 'str', 'weapon': 'str', 'quality': 'str', 'type': 'str', 'collection': 'str', 'link': 'str', 'price_series': 'list', 'price_ohlc': 'NoneType', 'price_last_update': 'int', 'keyfigures': 'dict'}


## Histórico de preços (price_series ou OHLC)

In [2]:
if item["price_series"]:
    df = (
        pl.DataFrame(item["price_series"])
          .with_columns(pl.from_epoch("ts", time_unit="s").alias("date"))
          .sort("date")
    )
elif item["price_ohlc"]:
    df = (
        pl.DataFrame(item["price_ohlc"])
          .with_columns(
              pl.from_epoch("ts", time_unit="ms").alias("date"),
              pl.col("c").alias("price"),
          )
          .sort("date")
    )
else:
    df = pl.DataFrame()

if df.height:
    dmin, dmax = df["date"].min(), df["date"].max()
    print(f"Pontos:        {df.height}")
    print(f"Primeira data: {dmin.date()}")
    print(f"Última data:   {dmax.date()}")
    print(f"Cobertura:     {(dmax - dmin).days} dias")
    gaps_h = df.select(pl.col("date").diff().dt.total_seconds() / 3600).drop_nulls().to_series()
    print(f"Gap médio:     {gaps_h.mean():.2f}h | mediano: {gaps_h.median():.2f}h | max: {gaps_h.max():.1f}h")
    print(f"Range preço:   {df['price'].min():.2f} → {df['price'].max():.2f}")
df.tail()

Pontos:        34504
Primeira data: 2014-02-21
Última data:   2026-05-30
Cobertura:     4481 dias
Gap médio:     3.12h | mediano: 1.00h | max: 24.0h
Range preço:   0.28 → 563.80


ts,price,volume,date
f64,f64,f64,datetime[μs]
1.7801e9,38.0,3.0,2026-05-30 07:00:00
1.7801e9,41.007,9.0,2026-05-30 08:00:00
1.7801e9,48.674,2.0,2026-05-30 09:00:00
1.7801e9,41.188,7.0,2026-05-30 10:00:00
1.7801e9,41.29,5.0,2026-05-30 11:00:00


In [3]:
import plotly.graph_objects as go

if item.get("price_ohlc"):
    raw = (
        pl.DataFrame(item["price_ohlc"])
          .with_columns(pl.from_epoch("ts", time_unit="ms").alias("date"))
    )
    fig = go.Figure(data=go.Candlestick(
        x=raw["date"].to_list(),
        open=raw["o"].to_list(),
        high=raw["h"].to_list(),
        low=raw["l"].to_list(),
        close=raw["c"].to_list(),
    ))
else:
    fig = go.Figure(go.Scatter(
        x=df["date"].to_list(),
        y=df["price"].to_list(),
        mode="lines",
        name="price",
    ))
fig.update_layout(title=ITEM_NAME, xaxis_title="Date", yaxis_title="Price (USD)", height=500)
fig.show()

## Preços atuais por marketplace

In [4]:
kf = item["keyfigures"]
if kf:
    marketplaces = ["steam", "skinport", "skinbaron", "csmoney", "skinswap", "haloskins", "uuskins"]
    rows = []
    for mp in marketplaces:
        d = kf.get(mp)
        if d:
            rows.append({
                "marketplace": mp,
                "price": d.get("current_price"),
                "volume": d.get("current_volume"),
                "link": d.get("offer_link"),
            })
    pl.DataFrame(rows).sort("price")
else:
    print("sem keyfigures para este item")

## Estatísticas (último mês / último ano)

In [5]:
if kf:
    m = kf["last_month_statistics"] or {}
    y = kf["last_year_statistics"] or {}
    keys = list(m.keys()) or list(y.keys())
    stats = pl.DataFrame({
        "metric": keys,
        "month": [m.get(k) for k in keys],
        "year": [y.get(k) for k in keys],
    })
    stats